# Week 3 — Symmetric Crypto & Hashes: modes and misuse

**Lesson plan:** [`../weeks/week-03.md`](../weeks/week-03.md)
**Reading:** the ECB-penguin example (Wikipedia's block-cipher-mode article is
fine) + a short HMAC/length-extension primer.

Pure Python, no lab target. Weeks 1–2 were about *ciphers*; this week is about
**how you use them** — because modern ciphers (AES) are strong, and almost every
real break is a **misuse of the mode**, not a break of the cipher.

> ### The one idea
> AES is not the thing that fails. **The mode is.** Encrypt with ECB and structure
> leaks; reuse a CTR nonce and you're back to week 2's two-time pad; build a MAC as
> `H(secret‖msg)` and an attacker forges tags without the key. The cipher kept its
> promise; the *construction around it* broke.

*These demos use a small stand-in hash as the block primitive so the notebook is
self-contained and stdlib-only. The **properties** shown — determinism, keystream
reuse, resumable Merkle–Damgård state — are exactly those of real AES modes and
SHA-2; the lessons transfer verbatim.*

## 1 · ECB — encrypting each block independently leaks structure

Electronic Code Book encrypts every block on its own. A block cipher is a keyed
*permutation*: the same input block always maps to the same output block. So
**identical plaintext blocks produce identical ciphertext blocks** — and any
structure in the plaintext (a flat region of an image, repeated records) survives
into the ciphertext. That's the famous "ECB penguin."

In [ ]:
import hashlib, os

def block_cipher(block, key):
    """Stand-in for a real block cipher: a deterministic keyed map on a block.
    Real AES is a keyed permutation; the only property we need here is that it is
    DETERMINISTIC in the block (identical block -> identical output)."""
    return hashlib.sha256(key + block).digest()[:len(block)]

BS = 3

def ecb_encrypt(data, key):
    data += bytes((-len(data)) % BS)                       # pad to block size
    return b"".join(block_cipher(data[i:i+BS], key) for i in range(0, len(data), BS))

def distinct_blocks(data):
    return len({data[i:i+BS] for i in range(0, len(data), BS)})

key = os.urandom(16)
# An "image" with two flat regions (repeated byte-triples), like the penguin's body.
image = (bytes([65]) * 36 + bytes([66]) * 36) * 4
ecb_ct = ecb_encrypt(image, key)

print(f"plaintext: {distinct_blocks(image)} distinct blocks (2 flat regions)")
print(f"ECB ciphertext: {distinct_blocks(ecb_ct)} distinct blocks")
print("-> the block structure is preserved 1:1. Flat stays flat. The penguin shows.")

## 2 · CBC — chaining hides the structure

Cipher Block Chaining XORs each plaintext block with the *previous ciphertext
block* before encrypting. Now identical plaintext blocks encrypt differently
(their predecessors differ), so structure vanishes.

In [ ]:
def cbc_encrypt(data, key, iv):
    data += bytes((-len(data)) % BS)
    prev, out = iv, b""
    for i in range(0, len(data), BS):
        x = bytes(a ^ b for a, b in zip(data[i:i+BS], prev))
        prev = block_cipher(x, key)
        out += prev
    return out

cbc_ct = cbc_encrypt(image, key, os.urandom(BS))
print(f"CBC ciphertext: {distinct_blocks(cbc_ct)} distinct blocks "
      f"(vs {distinct_blocks(image)} in the plaintext)")
print("-> same image, same cipher, different MODE. Structure hidden.")
print("""
The lesson isn't 'use CBC' (it has its own pitfalls — padding oracles, week 7-ish).
It's that the MODE, not the cipher, decided whether your data leaked. ECB is never
the right answer for structured data, and 'we used AES' tells you nothing until you
know the mode.""")

## 3 · ⚠️ CTR nonce reuse — week 2's two-time pad, in a modern mode

Counter mode turns a block cipher into a stream cipher: it generates a keystream
from `cipher(key, nonce‖counter)` and XORs it with the plaintext. This is how AES
is usually used (AES-CTR, AES-GCM). It is excellent — **until you reuse a nonce
under the same key.** Then two messages share a keystream, and:

# C₁ ⊕ C₂ = P₁ ⊕ P₂

the exact two-time-pad break from week 2. The strong modern cipher gives you *zero*
protection against this; the guarantee was conditional on nonce uniqueness.

In [ ]:
def ctr_keystream(key, nonce, length):
    ks, ctr = b"", 0
    while len(ks) < length:
        ks += hashlib.sha256(key + nonce + ctr.to_bytes(4, "big")).digest()
        ctr += 1
    return ks[:length]

def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

nonce = os.urandom(8)                 # the BUG: this same nonce is used twice
m1 = b"transfer 1000 dollars to account 4471xy"
m2 = b"the quarterly meeting moved to three pm"
ks = ctr_keystream(key, nonce, max(len(m1), len(m2)))
c1, c2 = xor(m1, ks), xor(m2, ks)

print("nonce reused across two messages under the same key.")
print("attacker computes c1 XOR c2 — keystream cancels:")
print("  c1 XOR c2 == m1 XOR m2 ?", xor(c1, c2) == xor(m1, m2))

# crib-drag a guessed word, exactly as in week 2
x = xor(c1, c2)
crib = b"transfer"
for i in range(len(x) - len(crib)):
    frag = xor(x[i:i+len(crib)], crib)
    if all(c == 32 or 97 <= c <= 122 for c in frag):
        print(f"  crib {crib!r} at pos {i}: other message reads {frag!r}")
print("""
AES-CTR with a unique nonce every time: strong. AES-CTR with a repeated nonce: a
two-time pad. Same code you wrote in week 2 breaks it. 'We use AES-256' is not a
security claim until you also promise the nonce is never reused.""")

## 4 · Hashes and the length-extension trap

A tempting way to authenticate a message with a secret: `tag = H(secret ‖ message)`.
Only someone with the secret could compute it, right? **Wrong**, if H is a
Merkle–Damgård hash (MD5, SHA-1, SHA-256): the digest *is the full internal state*,
so an attacker who has one tag can **resume hashing** and forge a tag for
`message ‖ padding ‖ anything` — without ever learning the secret.

In [ ]:
def compress(state, block):
    """Toy Merkle-Damgard compression: state = f(state, block). The point is that
    the hash's OUTPUT is this state, so hashing can be RESUMED from a digest."""
    s = state
    for b in block:
        s = ((s * 31) + b) & 0xFFFFFFFF
    return s

def md_hash(msg, iv=0x12345678):
    msg += bytes((-len(msg)) % 4)
    s = iv
    for i in range(0, len(msg), 4):
        s = compress(s, msg[i:i+4])
    return s

def bad_mac(secret, msg):
    return md_hash(secret + msg)          # the vulnerable construction

secret = b"s3cr3tk"                        # attacker does NOT know this
msg = b"amount=100&to=alice"
tag = bad_mac(secret, msg)                 # attacker observes (msg, tag)
print(f"legit tag over {msg!r}: {tag:#010x}")

# FORGE: append data and produce a valid tag, knowing only (msg, tag, len(secret)).
extension = b"&to=attacker"
total = len(secret) + len(msg)
pad = bytes((-total) % 4)                   # replicate the hash's internal padding
forged_msg = msg + pad + extension          # what the server will verify
forged_tag = md_hash(extension, iv=tag)     # RESUME hashing from the observed tag

server_tag = bad_mac(secret, forged_msg)    # what the server computes
print(f"forged msg: {forged_msg!r}")
print(f"forged tag matches server's tag? {forged_tag == server_tag}")
assert forged_tag == server_tag
print("-> a valid MAC for attacker-chosen data, forged WITHOUT the secret key.")

## 5 · The fix — HMAC is not length-extendable

HMAC nests the hashing: `HMAC(k, m) = H((k ⊕ opad) ‖ H((k ⊕ ipad) ‖ m))`. The
attacker never sees the inner state that produced the final output, so they can't
resume it. The extension attack simply doesn't apply.

In [ ]:
import hmac, hashlib as h

def good_mac(secret, msg):
    return hmac.new(secret, msg, h.sha256).hexdigest()

secret = b"s3cr3tk"
msg = b"amount=100&to=alice"
tag = good_mac(secret, msg)
print("HMAC tag:", tag[:32], "...")

# The attacker cannot resume from this tag: it is H(outer ‖ H(inner ‖ msg)), and the
# inner state is never exposed. Verify a naive "extend" fails to validate.
extension = b"&to=attacker"
forged_guess = good_mac(b"", msg + extension)   # attacker has no valid path
print("naive forged tag validates?", hmac.compare_digest(forged_guess,
                                                          good_mac(secret, msg + extension)))
print("""
HMAC (real hashlib this time) closes the hole structurally, not by patching. The
week's pattern: the primitive (SHA-256) was fine; the naive CONSTRUCTION H(k‖m) was
the bug; the fix is a better construction (HMAC), not a better hash.""")

## 6 · The scorecard view — the cipher isn't the control, the construction is

| Construction | Guarantee (axis 2) | Its condition / failure |
|---|---|---|
| ECB mode | confidentiality of individual blocks only | **leaks structure** — identical blocks visible |
| CBC / CTR | confidentiality of the message | CTR: **nonce must never repeat** (else two-time pad) |
| `H(secret‖msg)` MAC | *appears* to authenticate | **length extension** forges tags without the key |
| HMAC | authentication, no length-extension | needs a secret key; that's it |

> Every failure this week is a **misuse**, not a broken primitive. "We use AES /
> SHA-256" is the answer to a question nobody asked; the security lives in the mode,
> the nonce discipline, and the construction. Naming the condition (week 1's habit)
> is again the whole skill — *AES-CTR is confidential PROVIDED nonces never
> repeat.*

## 7 · Your studio deliverable

In `week03/`:

1. **ECB vs CBC** — encrypt a structured input in both; show ECB preserves block
   structure and CBC destroys it. (Bonus: render an actual image with matplotlib to
   see the penguin.)
2. **CTR nonce reuse** — reuse a nonce across two messages and recover both by
   crib-dragging. Explicitly connect it to the week-2 two-time pad.
3. **Length extension** — forge a valid `H(secret‖msg)` tag for extended data
   without the secret; then show HMAC defeats the same attempt.
4. **Control Scorecard** — for each construction, state the guarantee **and its
   condition**, and classify the failure as a primitive break vs. a misuse. (They
   are all misuses — that's the point.)